# Replicação da triagem cientométrica — previsão de enchentes com IA/ML
**Dissertação de mestrado (PPGCA/UNIVALI) · Capítulo 3 (artigo de revisão/cientometria)**

Este notebook **reexecuta, sem alterar nenhuma regra, os scripts originais** que produziram o corpus de 2.068 artigos e os estratos Núcleo A/B/Periférico/Descarte (janeiro/2026), e depois **testa o método**: confere se os números reproduzem, expõe as falhas encontradas e mede a robustez do resultado.

**Importante para quem avalia**
- A triagem é **automática** (casamento de termos + escore ponderado). **Não houve revisão manual registro a registro**, portanto não existe concordância entre revisores (κ).
- Os escores 0–100 são **relativos ao corpus** (min–max). O "100" não é qualidade absoluta, e o corte de 70 depende da composição do corpus (ver seção 7).
- Os scripts das seções 3–4 são os originais (Fase 1 reescrita de forma equivalente; Fase 5 na versão corrigida). Adaptações são marcadas com `# [ADAPTAÇÃO]`.

## Como executar
1. Rode a célula da seção 1 e informe os **8 arquivos CSV** exportados das bases (as exportações têm licença das bases e não são redistribuídas aqui; as strings de busca estão no repositório `cientometria`).
2. `Ambiente de execução → Executar tudo`.
3. A seção 5 imprime ✓/✗ comparando cada número com o relatório original.

| Arquivo esperado | Base |
|---|---|
| `lens-export.csv`, `lens-export-port.csv` | Lens (EN / PT) |
| `scopus-sem-filtro.csv`, `scopus-com-filtro.csv`, `scopus-port.csv` | Scopus (duas exportações da mesma busca + PT) |
| `ieee-export.csv` | IEEE Xplore |
| `SciELO.csv`, `SciELO-port.csv` | SciELO |

## 1. Dados de entrada

In [ ]:
import os
# Opção A (padrão): enviar os 8 CSVs do computador.
# Opção B: se os arquivos estão no seu Google Drive, defina MODO='drive' e ajuste PASTA_DRIVE.
MODO = 'upload'
PASTA_DRIVE = '/content/drive/MyDrive/CAMINHO/DA/PASTA/COM/OS/CSVs'
BASES_DIR = '/content/bases'
ESPERADOS = ['lens-export.csv','lens-export-port.csv','scopus-sem-filtro.csv','scopus-com-filtro.csv',
             'scopus-port.csv','ieee-export.csv','SciELO.csv','SciELO-port.csv']
if MODO == 'drive':
    from google.colab import drive; drive.mount('/content/drive'); BASES_DIR = PASTA_DRIVE
else:
    os.makedirs(BASES_DIR, exist_ok=True)
    faltam = [f for f in ESPERADOS if not os.path.exists(os.path.join(BASES_DIR, f))]
    if faltam:
        from google.colab import files
        up = files.upload()
        for nome, conteudo in up.items(): open(os.path.join(BASES_DIR, nome), 'wb').write(conteudo)
print('Arquivos presentes:', [f for f in ESPERADOS if os.path.exists(os.path.join(BASES_DIR, f))])
print('Ausentes:', [f for f in ESPERADOS if not os.path.exists(os.path.join(BASES_DIR, f))])

## 2. Resultados declarados no relatório original (para conferência)
Valores extraídos do `relatorio_cientometria.md` / `FORMULAS_RESUMO.md` gerados em jan/2026.

In [ ]:
DECLARADO = {
  'registros nas exportações': 16845,
  'removidos por DOI (declarado)': 4233, 'removidos por título (declarado)': 1065,
  'removidos no total (declarado)': 5298,
  'únicos após deduplicação': 11547,
  'corpus (núcleo expandido)': 2068,
  'Núcleo A': 13, 'Núcleo B': 494, 'Periférico': 1508, 'Descarte': 53,
}
DECLARADO

## 3. Scripts originais — Fases 1 a 5
Executados na ordem em que foram gerados. **Não alteramos regras.** Pontos que o leitor deve observar enquanto lê o código:
- `normalize_database` atribui `source_db` a um `DataFrame` vazio → a coluna fica **vazia** (perde-se a proveniência).
- A deduplicação por DOI usa `df[df['doi'] != '']`, o que **descarta todos os registros sem DOI** (não é deduplicação).
- Os termos de exclusão (`'dam'`, `'dike'`, `'levee'`) são testados por **trecho de texto** (substring), portanto `'dam'` casa com *damage*.
- Colunas mapeadas que não existem em IEEE/SciELO/Scopus caem silenciosamente em `'Unknown'`/vazio.

### Fase 1 — Carregamento (reescrita equivalente: o original repete o mesmo bloco `try/except` oito vezes)

In [ ]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

print("="*80)
print("CIENTOMETRIA - ANÁLISE BIBLIOGRÁFICA PARA PREVISÃO DE ENCHENTES COM IA/ML")
print("="*80)
print("\n[FASE 1] CARREGAMENTO DE BASES DE DADOS\n")

os.chdir(BASES_DIR)  # [ADAPTAÇÃO] o original assumia os arquivos na pasta de trabalho
bases_loaded = {}
def _carregar(arquivo, rotulo):
    try:
        df = pd.read_csv(arquivo)
        bases_loaded[rotulo] = len(df)
        print(f"✓ {rotulo}: {len(df)} registros carregados")
        return df
    except Exception as e:
        print(f"✗ {rotulo}: Erro - {e}")
        return pd.DataFrame()

lens_en          = _carregar('lens-export.csv',        'Lens (EN)')
lens_pt          = _carregar('lens-export-port.csv',   'Lens (PT)')
scopus_all       = _carregar('scopus-sem-filtro.csv',  'Scopus (Sem Filtro)')
scopus_filtered  = _carregar('scopus-com-filtro.csv',  'Scopus (Com Filtro)')
scopus_pt        = _carregar('scopus-port.csv',        'Scopus (PT)')
ieee             = _carregar('ieee-export.csv',        'IEEE Xplore')
scielo_en        = _carregar('SciELO.csv',             'SciELO (EN)')
scielo_pt        = _carregar('SciELO-port.csv',        'SciELO (PT)')

print("\n" + "="*80); print("RESUMO DE CARREGAMENTO"); print("="*80)
for base, count in bases_loaded.items(): print(f"  {base:.<35} {count:>6} artigos")
print(f"  {'TOTAL':.<35} {sum(bases_loaded.values()):>6} artigos")

### Fase 2 — Normalização, harmonização e deduplicação

In [ ]:
# ============================================================================
# FASE 2: NORMALIZAÇÃO E PREPARAÇÃO DE DADOS
# ============================================================================

print("\n" + "="*80)
print("[FASE 2] NORMALIZAÇÃO E HARMONIZAÇÃO DE CAMPOS")
print("="*80 + "\n")

# Função para inspecionar campos disponíveis
def inspect_dataframe(df, name, nrows=3):
    print(f"\n{name}")
    print(f"  Dimensões: {df.shape}")
    if len(df) > 0:
        print(f"  Colunas: {list(df.columns[:10])}...")
    return df

# Inspeção breve
lens_en = inspect_dataframe(lens_en, "Lens (EN)")
scopus_all = inspect_dataframe(scopus_all, "Scopus (Sem Filtro)")
ieee = inspect_dataframe(ieee, "IEEE")
scielo_en = inspect_dataframe(scielo_en, "SciELO (EN)")

print("\n" + "="*80)
print("[FASE 2a] MAPEAMENTO DE COLUNAS COMUNS")
print("="*80 + "\n")

# Mapeamento de campos-chave entre bases
field_mapping = {
    'lens': {
        'title': 'Title',
        'year': 'Publication Year',
        'abstract': 'Abstract',
        'keywords': 'Keywords',
        'citations': 'Citing Works Count',
        'doi': 'DOI',
        'source': 'Source Title',
        'field': 'Fields of Study'
    },
    'scopus': {
        'title': 'Title',
        'year': 'Year',
        'abstract': 'Abstract',
        'keywords': 'Keywords',
        'citations': 'Cited by',
        'doi': 'DOI',
        'source': 'Source title',
        'field': 'Document Type'
    },
    'ieee': {
        'title': 'Title',
        'year': 'Publication Year',
        'abstract': 'Abstract',
        'keywords': 'Keywords',
        'citations': 'Citations',
        'doi': 'DOI',
        'source': 'Source Title',
        'field': 'Content Type'
    },
    'scielo': {
        'title': 'Title',
        'year': 'Year',
        'abstract': 'Abstract',
        'keywords': 'Keywords',
        'citations': None,
        'doi': 'DOI',
        'source': 'Source',
        'field': 'Document Type'
    }
}

print("✓ Mapeamento de campos definido para 4 bases principais")

# Função para normalizar e consolidar dados
def normalize_database(df, base_name, mapping, db_type='lens'):
    """
    Normaliza um dataframe usando o mapeamento específico
    """
    if len(df) == 0:
        return pd.DataFrame()
    
    normalized = pd.DataFrame()
    normalized['source_db'] = base_name
    normalized['db_type'] = db_type
    
    # Título
    if mapping.get('title') in df.columns:
        normalized['title'] = df[mapping['title']].fillna('Unknown')
    else:
        normalized['title'] = 'Unknown'
    
    # Ano
    if mapping.get('year') in df.columns:
        try:
            normalized['year'] = pd.to_numeric(df[mapping['year']], errors='coerce').astype('Int64')
        except:
            normalized['year'] = None
    else:
        normalized['year'] = None
    
    # Abstract
    if mapping.get('abstract') in df.columns:
        normalized['abstract'] = df[mapping['abstract']].fillna('')
    else:
        normalized['abstract'] = ''
    
    # Keywords
    if mapping.get('keywords') in df.columns:
        normalized['keywords'] = df[mapping['keywords']].fillna('')
    else:
        normalized['keywords'] = ''
    
    # Citações
    if mapping.get('citations') and mapping['citations'] in df.columns:
        try:
            normalized['citations'] = pd.to_numeric(df[mapping['citations']], errors='coerce').fillna(0).astype(int)
        except:
            normalized['citations'] = 0
    else:
        normalized['citations'] = 0
    
    # DOI
    if mapping.get('doi') in df.columns:
        normalized['doi'] = df[mapping['doi']].fillna('')
    else:
        normalized['doi'] = ''
    
    # Fonte
    if mapping.get('source') in df.columns:
        normalized['source'] = df[mapping['source']].fillna('Unknown')
    else:
        normalized['source'] = 'Unknown'
    
    return normalized

print("\n✓ Função de normalização definida")

# Normalizar bases principais
print("\nNormalizando bases...\n")

norm_lens_en = normalize_database(lens_en, 'Lens (EN)', field_mapping['lens'], 'lens')
print(f"  Lens (EN) normalizada: {len(norm_lens_en)} registros")

norm_lens_pt = normalize_database(lens_pt, 'Lens (PT)', field_mapping['lens'], 'lens')
print(f"  Lens (PT) normalizada: {len(norm_lens_pt)} registros")

norm_scopus_all = normalize_database(scopus_all, 'Scopus (Sem Filtro)', field_mapping['scopus'], 'scopus')
print(f"  Scopus (Sem Filtro) normalizada: {len(norm_scopus_all)} registros")

norm_scopus_filtered = normalize_database(scopus_filtered, 'Scopus (Com Filtro)', field_mapping['scopus'], 'scopus')
print(f"  Scopus (Com Filtro) normalizada: {len(norm_scopus_filtered)} registros")

norm_scopus_pt = normalize_database(scopus_pt, 'Scopus (PT)', field_mapping['scopus'], 'scopus')
print(f"  Scopus (PT) normalizada: {len(norm_scopus_pt)} registros")

norm_ieee = normalize_database(ieee, 'IEEE Xplore', field_mapping['ieee'], 'ieee')
print(f"  IEEE Xplore normalizada: {len(norm_ieee)} registros")

norm_scielo_en = normalize_database(scielo_en, 'SciELO (EN)', field_mapping['scielo'], 'scielo')
print(f"  SciELO (EN) normalizada: {len(norm_scielo_en)} registros")

norm_scielo_pt = normalize_database(scielo_pt, 'SciELO (PT)', field_mapping['scielo'], 'scielo')
print(f"  SciELO (PT) normalizada: {len(norm_scielo_pt)} registros")

# Consolidar em uma base única, removendo duplicatas
print("\n" + "="*80)
print("[FASE 2b] CONSOLIDAÇÃO E DEDUPLICAÇÃO")
print("="*80 + "\n")

all_dfs = [
    norm_lens_en, norm_lens_pt,
    norm_scopus_all, norm_scopus_filtered, norm_scopus_pt,
    norm_ieee,
    norm_scielo_en, norm_scielo_pt
]

# Remover DataFrames vazios
all_dfs = [df for df in all_dfs if len(df) > 0]

# Consolidar
df_consolidated = pd.concat(all_dfs, ignore_index=True)

print(f"Registros antes de deduplicação: {len(df_consolidated)}")

# Deduplicar por DOI (se disponível) e depois por título
df_consolidated['title_normalized'] = df_consolidated['title'].str.lower().str.strip()

# Primeiro, remover por DOI
df_consolidated = df_consolidated[df_consolidated['doi'] != ''].drop_duplicates(subset=['doi'], keep='first')
print(f"Após remover DOIs duplicados: {len(df_consolidated)}")

# Depois, remover por título (use case-insensitive)
df_consolidated = df_consolidated.drop_duplicates(subset=['title_normalized'], keep='first')
print(f"Após remover títulos duplicados: {len(df_consolidated)}")

print(f"\n✓ Base consolidada: {len(df_consolidated)} artigos únicos")

# Resumo estatístico básico
print("\n" + "="*80)
print("RESUMO DESCRITIVO DA BASE CONSOLIDADA")
print("="*80)
print(f"\nArtigos por fonte:")
print(df_consolidated['source_db'].value_counts().to_string())
print(f"\nPeríodo de publicação: {df_consolidated['year'].min():.0f} - {df_consolidated['year'].max():.0f}")
print(f"Anos com dados: {df_consolidated['year'].notna().sum()}")
print(f"Artigos com Abstract: {(df_consolidated['abstract'] != '').sum()}")
print(f"Artigos com Palavras-chave: {(df_consolidated['keywords'] != '').sum()}")
print(f"Artigos com DOI: {(df_consolidated['doi'] != '').sum()}")
print(f"Citações (média): {df_consolidated['citations'].mean():.2f}")
print(f"Citações (mediana): {df_consolidated['citations'].median():.0f}")

### Fase 3 — Filtro temático (listas de termos, casamento por trecho de texto)

In [ ]:
# ============================================================================
# FASE 3: DEFINIÇÃO DE CRITÉRIOS TEMÁTICOS BASEADOS NA INTENÇÃO DE PESQUISA
# ============================================================================

print("\n" + "="*80)
print("[FASE 3] FILTRAGEM TEMÁTICA - DEFINIÇÃO DE CRITÉRIOS DE ADERÊNCIA")
print("="*80 + "\n")

# Critérios extraídos da sua dissertação (arquivo 8 e 12)
print("CONTEXTO DA DISSERTAÇÃO (Daiane Wan-Dall Splitter da Silva):")
print("  Objetivo: Modelo preditivo para previsão de enchentes em comunidades costeiras")
print("  Foco: Santa Catarina, Brasil")
print("  Dados de entrada: Satélites, sensores, registros históricos de enchentes")
print("  Algoritmos esperados: LSTM, RNN, redes multicamadas, ensemble methods")
print("  Saídas: Previsão de risco, mapas de suscetibilidade, early warning")
print("\n")

# Definição de termos-chave por categoria
keywords_core = {
    'flood_prediction': [
        'flood prediction', 'flood forecast', 'flood early warning',
        'flood forecasting', 'inundation prediction', 'flood susceptibility',
        'flood risk', 'flood modeling', 'flood hazard', 'flooding',
        'enchentes', 'previsão de enchentes', 'previsão de inundações',
        'risco de inundação', 'modelagem de inundações', 'inundação urbana',
        'alerta de cheias', 'mapeamento de risco de cheias'
    ],
    'ai_ml_methods': [
        'machine learning', 'deep learning', 'lstm', 'rnn', 'neural network',
        'random forest', 'ensemble', 'support vector machine', 'xgboost',
        'artificial intelligence', 'data-driven', 'convolutional', 'cnn',
        'artificial neural', 'gradient boosting', 'supervised learning',
        'aprendizado de máquina', 'aprendizado profundo', 'rede neural',
        'inteligência artificial'
    ],
    'hydrological_data': [
        'rainfall', 'precipitation', 'water level', 'streamflow', 'discharge',
        'river', 'hydrological', 'hydrologic', 'basin', 'watershed',
        'gauge', 'stream', 'flow rate', 'inundation depth', 'inundation extent',
        'chuva', 'precipitação', 'nível de água', 'vazão', 'hidrológico'
    ],
    'geospatial_data': [
        'satellite', 'remote sensing', 'dem', 'digital elevation', 'landsat',
        'sentinel', 'geospatial', 'spatial', 'gis', 'raster', 'urban',
        'land cover', 'land use', 'lulc', 'ndvi', 'radar',
        'imagem satélite', 'sensoriamento remoto', 'terreno', 'mapa'
    ]
}

# Critérios de exclusão
keywords_exclude = [
    'flood myth', 'flood geology', 'flood insurance', 'flood psychology',
    'flood relief', 'flood control structure', 'dam', 'dike', 'levee',
    'flood insurance rate', 'flood mitigation infrastructure', 'green infrastructure',
    'semantic flooding', 'buffer overflow', 'network flooding'
]

print("TERMOS-CHAVE POR CATEGORIA:\n")
for cat, terms in keywords_core.items():
    print(f"  {cat.upper()}: {len(terms)} termos")
    print(f"    Exemplos: {', '.join(terms[:5])}...")

print(f"\n  TERMOS DE EXCLUSÃO: {len(keywords_exclude)} termos")
print(f"    Exemplos: {', '.join(keywords_exclude[:5])}...")

# Função de scoring temático
def check_keyword_presence(text, keywords_list, match_type='any'):
    """
    Verifica presença de palavras-chave em um texto
    match_type: 'any' (pelo menos 1), 'multiple' (múltiplas)
    """
    if pd.isna(text) or text == '':
        return 0
    
    text_lower = str(text).lower()
    matches = 0
    for kw in keywords_list:
        if kw.lower() in text_lower:
            matches += 1
    
    if match_type == 'any':
        return 1 if matches >= 1 else 0
    elif match_type == 'multiple':
        return 1 if matches >= 2 else 0
    return matches

def check_exclusion(text, keywords_exclude):
    """
    Verifica se o texto contém termos de exclusão
    """
    if pd.isna(text) or text == '':
        return False
    
    text_lower = str(text).lower()
    for kw in keywords_exclude:
        if kw.lower() in text_lower:
            return True
    return False

print("\n" + "="*80)
print("[FASE 3a] APLICAÇÃO DE FILTROS TEMÁTICOS")
print("="*80 + "\n")

# Criar campos de scoring temático
df_consolidated['flood_pred_score'] = df_consolidated.apply(
    lambda row: check_keyword_presence(
        str(row['title']) + ' ' + str(row['abstract']) + ' ' + str(row['keywords']),
        keywords_core['flood_prediction'],
        'any'
    ), axis=1
)

df_consolidated['ai_ml_score'] = df_consolidated.apply(
    lambda row: check_keyword_presence(
        str(row['title']) + ' ' + str(row['abstract']) + ' ' + str(row['keywords']),
        keywords_core['ai_ml_methods'],
        'any'
    ), axis=1
)

df_consolidated['hydro_score'] = df_consolidated.apply(
    lambda row: check_keyword_presence(
        str(row['abstract']) + ' ' + str(row['keywords']),
        keywords_core['hydrological_data'],
        'any'
    ), axis=1
)

df_consolidated['geospatial_score'] = df_consolidated.apply(
    lambda row: check_keyword_presence(
        str(row['abstract']) + ' ' + str(row['keywords']),
        keywords_core['geospatial_data'],
        'any'
    ), axis=1
)

df_consolidated['is_excluded'] = df_consolidated.apply(
    lambda row: check_exclusion(
        str(row['title']) + ' ' + str(row['abstract']),
        keywords_exclude
    ), axis=1
)

print("✓ Scores temáticos calculados para todos os artigos\n")

# Calcular composição temática
print("DISTRIBUIÇÃO DE SCORES TEMÁTICOS:\n")
print(f"  Artigos com 'flood prediction': {df_consolidated['flood_pred_score'].sum()} ({df_consolidated['flood_pred_score'].sum()/len(df_consolidated)*100:.1f}%)")
print(f"  Artigos com 'AI/ML': {df_consolidated['ai_ml_score'].sum()} ({df_consolidated['ai_ml_score'].sum()/len(df_consolidated)*100:.1f}%)")
print(f"  Artigos com dados 'hidrológicos': {df_consolidated['hydro_score'].sum()} ({df_consolidated['hydro_score'].sum()/len(df_consolidated)*100:.1f}%)")
print(f"  Artigos com dados 'geoespaciais': {df_consolidated['geospatial_score'].sum()} ({df_consolidated['geospatial_score'].sum()/len(df_consolidated)*100:.1f}%)")
print(f"  Artigos EXCLUÍDOS por palavras negativas: {df_consolidated['is_excluded'].sum()}")

# CRITÉRIO 1: Núcleo obrigatório
print("\n" + "="*80)
print("[FILTRO 1] NÚCLEO OBRIGATÓRIO (Flood Prediction + AI/ML + NOT Excluded)")
print("="*80 + "\n")

df_core = df_consolidated[
    (df_consolidated['flood_pred_score'] == 1) &
    (df_consolidated['ai_ml_score'] == 1) &
    (df_consolidated['is_excluded'] == False)
].copy()

print(f"Artigos no NÚCLEO OBRIGATÓRIO: {len(df_core)} ({len(df_core)/len(df_consolidated)*100:.1f}%)\n")

# CRITÉRIO 2: Núcleo expandido (permite um OU outro em dados)
print("="*80)
print("[FILTRO 2] NÚCLEO EXPANDIDO (FP + AI/ML + (Hydro OR Geospatial) + NOT Excluded)")
print("="*80 + "\n")

df_expanded = df_consolidated[
    (df_consolidated['flood_pred_score'] == 1) &
    (df_consolidated['ai_ml_score'] == 1) &
    ((df_consolidated['hydro_score'] == 1) | (df_consolidated['geospatial_score'] == 1)) &
    (df_consolidated['is_excluded'] == False)
].copy()

print(f"Artigos no NÚCLEO EXPANDIDO: {len(df_expanded)} ({len(df_expanded)/len(df_consolidated)*100:.1f}%)\n")

# CRITÉRIO 3: Periférico (apenas FP + AI/ML, sem necessidade de dados específicos)
print("="*80)
print("[FILTRO 3] PERIFÉRICO (FP + AI/ML + NOT Excluded, sem restrição de dados)")
print("="*80 + "\n")

df_peripheral = df_consolidated[
    (df_consolidated['flood_pred_score'] == 1) &
    (df_consolidated['ai_ml_score'] == 1) &
    (df_consolidated['is_excluded'] == False)
].copy()

print(f"Artigos no PERIFÉRICO: {len(df_peripheral)} ({len(df_peripheral)/len(df_consolidated)*100:.1f}%)\n")

print("="*80)
print("RESUMO DE FILTROS TEMÁTICOS")
print("="*80)
print(f"  Base original....................: {len(df_consolidated):>5} artigos")
print(f"  - Núcleo obrigatório (criterioso): {len(df_core):>5} artigos ({len(df_core)/len(df_consolidated)*100:.1f}%)")
print(f"  - Núcleo expandido (moderado)....: {len(df_expanded):>5} artigos ({len(df_expanded)/len(df_consolidated)*100:.1f}%)")
print(f"  - Periférico (liberal)...........: {len(df_peripheral):>5} artigos ({len(df_peripheral)/len(df_consolidated)*100:.1f}%)")
print("="*80)

### Fase 4 — Escore de relevância (C\*, R\*, T\*, A\*) e normalização min–max

In [ ]:
# ============================================================================
# FASE 4: CÁLCULO DE SCORES DE RELEVÂNCIA COM EQUAÇÕES EXPLÍCITAS
# ============================================================================

print("\n" + "="*80)
print("[FASE 4] CÁLCULO DE SCORES DE RELEVÂNCIA BIBLIOMÉTRICOS")
print("="*80 + "\n")

# Usar o Núcleo Expandido como ponto de partida (balanço entre rigor e viabilidade)
df_analysis = df_expanded.copy()

print(f"Base para análise de score: {len(df_analysis)} artigos\n")

print("="*80)
print("COMPONENTES DO SCORE DE RELEVÂNCIA")
print("="*80 + "\n")

# 1. SCORE DE CITAÇÕES NORMALIZADO (C*)
print("1. SCORE DE CITAÇÕES NORMALIZADAS (C*)\n")

max_citations = df_analysis['citations'].max()
min_citations = df_analysis['citations'].min()

print(f"   Citações - Min: {min_citations}, Max: {max_citations}, Média: {df_analysis['citations'].mean():.1f}")
print(f"\n   Equação: C*_i = (C_i - min) / (max - min)")
print(f"   Onde C_i = número de citações do artigo i\n")

df_analysis['C_norm'] = (df_analysis['citations'] - min_citations) / (max_citations - min_citations) if max_citations > min_citations else 0
df_analysis['C_norm'] = df_analysis['C_norm'].fillna(0)

print(f"   ✓ C* calculado (intervalo: 0.0 a 1.0)")

# 2. SCORE DE RECÊNCIA (R*)
print("\n2. SCORE DE RECÊNCIA (R*)\n")

max_year = df_analysis['year'].max()
min_year = df_analysis['year'][df_analysis['year'].notna()].min()

print(f"   Período - Min: {min_year:.0f}, Max: {max_year:.0f}")
print(f"\n   Equação: R*_i = (Y_i - min_year) / (max_year - min_year)")
print(f"   Onde Y_i = ano de publicação do artigo i\n")

# Tratar anos missing
df_analysis['R_norm'] = df_analysis['year'].apply(
    lambda y: (y - min_year) / (max_year - min_year) if pd.notna(y) and y >= min_year else 0
)

print(f"   ✓ R* calculado (intervalo: 0.0 a 1.0)")

# 3. SCORE DE ADERÊNCIA TEMÁTICA (T*)
print("\n3. SCORE DE ADERÊNCIA TEMÁTICA (T*)\n")

print("   Categorias:")
print("   - Categoria A (máxima aderência): título + abstract + keywords com termos flood prediction E AI/ML")
print("   - Categoria B (aderência moderada): título ou abstract com termos flood prediction E AI/ML")
print("   - Categoria C (aderência baixa): apenas uma categoria de busca presente\n")

def classify_thematic_adherence(row):
    """
    Classifica a aderência temática em 3 níveis
    """
    # Montar texto para cada campo
    title_abstract_kw = str(row['title']) + ' ' + str(row['abstract']) + ' ' + str(row['keywords'])
    title = str(row['title'])
    
    # Contar ocorrências em título vs resto
    flood_in_title = check_keyword_presence(title, keywords_core['flood_prediction'])
    ai_in_title = check_keyword_presence(title, keywords_core['ai_ml_methods'])
    
    flood_in_content = check_keyword_presence(title_abstract_kw, keywords_core['flood_prediction'])
    ai_in_content = check_keyword_presence(title_abstract_kw, keywords_core['ai_ml_methods'])
    
    # Categoria A: máxima (flood + AI no título ou em multiple campos)
    if (flood_in_title and ai_in_title):
        return 1.0
    # Categoria B: moderada (flood + AI em abstract/keywords)
    elif (flood_in_content and ai_in_content):
        return 0.7
    # Categoria C: baixa (um dos dois)
    else:
        return 0.4

df_analysis['T_score'] = df_analysis.apply(classify_thematic_adherence, axis=1)

print(f"   Distribuição de aderência temática:")
print(f"   - Categoria A (T*=1.0): {(df_analysis['T_score'] == 1.0).sum()} artigos ({(df_analysis['T_score'] == 1.0).sum()/len(df_analysis)*100:.1f}%)")
print(f"   - Categoria B (T*=0.7): {(df_analysis['T_score'] == 0.7).sum()} artigos ({(df_analysis['T_score'] == 0.7).sum()/len(df_analysis)*100:.1f}%)")
print(f"   - Categoria C (T*=0.4): {(df_analysis['T_score'] == 0.4).sum()} artigos ({(df_analysis['T_score'] == 0.4).sum()/len(df_analysis)*100:.1f}%)\n")

print(f"   ✓ T* calculado (intervalo: 0.4 a 1.0)")

# 4. SCORE DE ADERÊNCIA AOS DADOS/MÉTODO/RESULTADO (A*)
print("\n4. SCORE DE ADERÊNCIA AOS DADOS/MÉTODO/RESULTADO (A*)\n")

print("   Componentes:")
print("   - Input (I): presença de dados hidrológicos OU geoespaciais")
print("   - Method (M): presença de termos de ML/DL explícitos")
print("   - Output (O): previsão/forecasting/early warning/mapping")
print("   - Context (C): aplicação em inundações urbanas ou costeiras\n")

print("   Equação: A*_i = (I_i + M_i + O_i + C_i) / 4")
print("   Onde cada termo ∈ {0.0, 0.5, 1.0}\n")

# Input scoring
input_keywords = keywords_core['hydrological_data'] + keywords_core['geospatial_data']
df_analysis['input_score'] = df_analysis.apply(
    lambda row: (1.0 if check_keyword_presence(str(row['abstract']) + ' ' + str(row['keywords']), input_keywords) 
                 else 0.0), axis=1
)

# Method scoring (must have explicit ML/DL terms)
ml_specific = ['lstm', 'rnn', 'cnn', 'deep learning', 'neural network', 'machine learning', 
               'ensemble', 'random forest', 'support vector', 'xgboost', 'aprendizado de máquina',
               'rede neural', 'aprendizado profundo']
df_analysis['method_score'] = df_analysis.apply(
    lambda row: (1.0 if check_keyword_presence(str(row['abstract']) + ' ' + str(row['title']), ml_specific) 
                 else 0.5), axis=1
)

# Output scoring (prediction, forecasting, early warning, mapping)
output_keywords = ['prediction', 'forecast', 'early warning', 'risk mapping', 'susceptibility', 
                   'mapping', 'previsão', 'mapeamento', 'alerta']
df_analysis['output_score'] = df_analysis.apply(
    lambda row: (1.0 if check_keyword_presence(str(row['title']) + ' ' + str(row['abstract']), output_keywords) 
                 else 0.5), axis=1
)

# Context scoring (urban/coastal/basin-specific)
context_keywords = ['urban', 'coastal', 'river', 'basin', 'watershed', 'city', 'communities',
                    'urbano', 'costeiro', 'comunidades', 'rio']
df_analysis['context_score'] = df_analysis.apply(
    lambda row: (1.0 if check_keyword_presence(str(row['abstract']) + ' ' + str(row['keywords']), context_keywords) 
                 else 0.5), axis=1
)

df_analysis['A_score'] = (df_analysis['input_score'] + df_analysis['method_score'] + 
                           df_analysis['output_score'] + df_analysis['context_score']) / 4

print(f"   Distribuição de aderência a dados/método/resultado:")
print(f"   - A* ≥ 0.9: {(df_analysis['A_score'] >= 0.9).sum()} artigos ({(df_analysis['A_score'] >= 0.9).sum()/len(df_analysis)*100:.1f}%)")
print(f"   - 0.7 ≤ A* < 0.9: {((df_analysis['A_score'] >= 0.7) & (df_analysis['A_score'] < 0.9)).sum()} artigos")
print(f"   - 0.5 ≤ A* < 0.7: {((df_analysis['A_score'] >= 0.5) & (df_analysis['A_score'] < 0.7)).sum()} artigos\n")

print(f"   ✓ A* calculado (intervalo: 0.5 a 1.0)")

# 5. CÁLCULO DO SCORE FINAL
print("\n" + "="*80)
print("5. SCORE DE RELEVÂNCIA FINAL")
print("="*80 + "\n")

w1, w2, w3, w4 = 0.35, 0.20, 0.30, 0.15  # pesos

print(f"Equação geral:")
print(f"\n   Score_i = w1·C*_i + w2·R*_i + w3·T*_i + w4·A*_i\n")
print(f"Onde:")
print(f"   C*_i = Citações normalizadas (peso w1 = {w1})")
print(f"   R*_i = Recência normalizada (peso w2 = {w2})")
print(f"   T*_i = Aderência temática (peso w3 = {w3})")
print(f"   A*_i = Aderência dados/método/resultado (peso w4 = {w4})")
print(f"   Total de pesos: {w1 + w2 + w3 + w4}\n")

df_analysis['score_final'] = (w1 * df_analysis['C_norm'] + 
                               w2 * df_analysis['R_norm'] + 
                               w3 * df_analysis['T_score'] + 
                               w4 * df_analysis['A_score'])

# Normalizar score final para intervalo 0-100
min_score = df_analysis['score_final'].min()
max_score = df_analysis['score_final'].max()
df_analysis['score_final_100'] = ((df_analysis['score_final'] - min_score) / (max_score - min_score) * 100).round(2)

print(f"✓ Score final calculado")
print(f"\nEstatísticas do Score Final (0-100):")
print(f"   Mínimo: {df_analysis['score_final_100'].min():.2f}")
print(f"   Máximo: {df_analysis['score_final_100'].max():.2f}")
print(f"   Média: {df_analysis['score_final_100'].mean():.2f}")
print(f"   Mediana: {df_analysis['score_final_100'].median():.2f}")
print(f"   Desvio padrão: {df_analysis['score_final_100'].std():.2f}")

### Fase 5 — Estratos (versão corrigida; a primeira versão falhava na ordenação de `value_counts`)

In [ ]:
# Corrigir o erro de sintaxe na ordenação

print("\n" + "="*80)
print("[FASE 5] FÁCIAS DE PRIORIDADE E CORTES BIBLIOGRÁFICOS")
print("="*80 + "\n")

# Definir limites de corte usando percentis
percentis = df_analysis['score_final_100'].quantile([0.25, 0.5, 0.75, 0.90])

print("PERCENTIS DE DISTRIBUIÇÃO DE SCORES:\n")
for p, val in percentis.items():
    print(f"   {p*100:.0f}º percentil: {val:.2f}")

# Definir fácias
CORTE_NUCLEOA = 70
CORTE_NUCLEOB = 50
CORTE_PERIFERICA = 30

print("\n" + "="*80)
print("DEFINIÇÃO DE FÁCIAS")
print("="*80 + "\n")

df_analysis['fascia'] = 'Descarte'

# Núcleo A: top tier
df_analysis.loc[(df_analysis['score_final_100'] >= CORTE_NUCLEOA), 'fascia'] = 'Núcleo A'

# Núcleo B: medium-high
df_analysis.loc[(df_analysis['score_final_100'] >= CORTE_NUCLEOB) & 
                (df_analysis['score_final_100'] < CORTE_NUCLEOA), 'fascia'] = 'Núcleo B'

# Periférico
df_analysis.loc[(df_analysis['score_final_100'] >= CORTE_PERIFERICA) & 
                (df_analysis['score_final_100'] < CORTE_NUCLEOB), 'fascia'] = 'Periférico'

print(f"FÁCIAS DEFINIDAS COM CRITÉRIOS:\n")
print(f"1. NÚCLEO A (Leitura Detalhada - PRIORITÁRIO)")
print(f"   Critério: Score ≥ {CORTE_NUCLEOA}")
print(f"   Justificativa: Artigos com altíssima aderência temática, relevância e citações")
print(f"   Expectativa: ~{(df_analysis['fascia'] == 'Núcleo A').sum()} artigos\n")

print(f"2. NÚCLEO B (Leitura Seletiva - IMPORTANTE)")
print(f"   Critério: {CORTE_NUCLEOB} ≤ Score < {CORTE_NUCLEOA}")
print(f"   Justificativa: Artigos de boa qualidade, com boa aderência mas menos influentes")
print(f"   Expectativa: ~{(df_analysis['fascia'] == 'Núcleo B').sum()} artigos\n")

print(f"3. PERIFÉRICO (Leitura Opcional - COMPLEMENTAR)")
print(f"   Critério: {CORTE_PERIFERICA} ≤ Score < {CORTE_NUCLEOB}")
print(f"   Justificativa: Artigos relevantes mas com fatores limitantes")
print(f"   Expectativa: ~{(df_analysis['fascia'] == 'Periférico').sum()} artigos\n")

print(f"4. DESCARTE (Não Recomendado - EXCLUIR)")
print(f"   Critério: Score < {CORTE_PERIFERICA}")
print(f"   Justificativa: Baixa relevância, foco marginal")
print(f"   Expectativa: ~{(df_analysis['fascia'] == 'Descarte').sum()} artigos\n")

# Resumo de distribuição
print("="*80)
print("RESUMO DE DISTRIBUIÇÃO POR FÁCIA")
print("="*80 + "\n")

fascias_order = ['Núcleo A', 'Núcleo B', 'Periférico', 'Descarte']

for fascia in fascias_order:
    count = (df_analysis['fascia'] == fascia).sum()
    pct = count / len(df_analysis) * 100
    print(f"  {fascia:.<30} {count:>5} artigos ({pct:>5.1f}%)")

print(f"  {'TOTAL':.<30} {len(df_analysis):>5} artigos")

# Análise detalhada por fácia
print("\n" + "="*80)
print("ANÁLISE DESCRITIVA POR FÁCIA")
print("="*80 + "\n")

for fascia in ['Núcleo A', 'Núcleo B', 'Periférico']:
    subset = df_analysis[df_analysis['fascia'] == fascia]
    if len(subset) > 0:
        print(f"\n{fascia.upper()}")
        print(f"  Quantidade: {len(subset)} artigos")
        print(f"  Score final - Mín: {subset['score_final_100'].min():.1f}, Máx: {subset['score_final_100'].max():.1f}, "
              f"Média: {subset['score_final_100'].mean():.1f}")
        print(f"  Citações - Mín: {subset['citations'].min():.0f}, Máx: {subset['citations'].max():.0f}, "
              f"Média: {subset['citations'].mean():.1f}")
        
        recent_years = subset[subset['year'] >= 2020].shape[0]
        print(f"  Artigos publicados após 2020: {recent_years} ({recent_years/len(subset)*100:.1f}%)")
        
        with_abstract = (subset['abstract'] != '').sum()
        print(f"  Artigos com abstract: {with_abstract} ({with_abstract/len(subset)*100:.1f}%)")

# Estatísticas de pesos por fácia
print("\n" + "="*80)
print("CONTRIBUIÇÃO MÉDIA DOS COMPONENTES DO SCORE POR FÁCIA")
print("="*80 + "\n")

for fascia in ['Núcleo A', 'Núcleo B', 'Periférico']:
    subset = df_analysis[df_analysis['fascia'] == fascia]
    if len(subset) > 0:
        print(f"\n{fascia.upper()}")
        contrib_c = (w1 * subset['C_norm'].mean())
        contrib_r = (w2 * subset['R_norm'].mean())
        contrib_t = (w3 * subset['T_score'].mean())
        contrib_a = (w4 * subset['A_score'].mean())
        
        print(f"  Contribuição C* (Citações):     {contrib_c:.3f} ({contrib_c*100:.1f}% do score)")
        print(f"  Contribuição R* (Recência):     {contrib_r:.3f} ({contrib_r*100:.1f}% do score)")
        print(f"  Contribuição T* (Temática):     {contrib_t:.3f} ({contrib_t*100:.1f}% do score)")
        print(f"  Contribuição A* (Dados/Método): {contrib_a:.3f} ({contrib_a*100:.1f}% do score)")
        print(f"  {'Total:':.<20} {contrib_c + contrib_r + contrib_t + contrib_a:.3f}")

### Fase 6 — Exportação e relatórios
As células abaixo são as originais. **Atenção:** os textos finais (`COMECE_AQUI.txt`, relatório, FAQ) contêm números **escritos à mão** no código (ex.: "5.298 removidos", "1.676 por exclusão", "Núcleo Expandido 2.068"), não calculados. Compare com a seção 5.

In [ ]:
# [ADAPTAÇÃO] gravar saídas em /content/saida (o original gravava na pasta de trabalho)
OUT='/content/saida'; os.makedirs(OUT, exist_ok=True); os.chdir(OUT)

In [ ]:
# ============================================================================
# FASE 6: EXPORTAÇÃO E RELATÓRIOS FINAIS
# ============================================================================

print("\n" + "="*80)
print("[FASE 6] GERAÇÃO DE RELATÓRIOS E EXPORTAÇÃO DE DADOS")
print("="*80 + "\n")

# Preparar dataframe para exportação
df_export = df_analysis[[
    'source_db', 'title', 'year', 'citations', 'doi', 'source',
    'C_norm', 'R_norm', 'T_score', 'A_score', 'score_final_100', 'fascia'
]].copy()

df_export = df_export.sort_values('score_final_100', ascending=False)

# Exportar CSV completo
print("Exportando dados para CSV...\n")

csv_file = 'cientometria_analise_completa.csv'
df_export.to_csv(csv_file, index=False, encoding='utf-8')
print(f"✓ Arquivo completo: {csv_file}")

# Exportar apenas Núcleo A e B
df_nucleos = df_export[df_export['fascia'].isin(['Núcleo A', 'Núcleo B'])].copy()
csv_nucleos = 'cientometria_nucleos_ab.csv'
df_nucleos.to_csv(csv_nucleos, index=False, encoding='utf-8')
print(f"✓ Arquivo Núcleos A+B: {csv_nucleos} ({len(df_nucleos)} artigos)")

# Exportar apenas Núcleo A
df_nucleoa = df_export[df_export['fascia'] == 'Núcleo A'].copy()
csv_nucleoa = 'cientometria_nucleoa_recomendado.csv'
df_nucleoa.to_csv(csv_nucleoa, index=False, encoding='utf-8')
print(f"✓ Arquivo Núcleo A: {csv_nucleoa} ({len(df_nucleoa)} artigos)")

print("\n" + "="*80)
print("TOP 20 ARTIGOS RECOMENDADOS (NÚCLEO A + B)")
print("="*80 + "\n")

top20 = df_export[df_export['fascia'].isin(['Núcleo A', 'Núcleo B'])].head(20).copy()
top20['rank'] = range(1, len(top20) + 1)

for idx, row in top20.iterrows():
    print(f"{row['rank']:2d}. [{row['fascia']}] {row['title'][:80]}...")
    print(f"    Citações: {row['citations']:.0f} | Score: {row['score_final_100']:.1f} | Ano: {row['year']:.0f}")
    print()

print("\n" + "="*80)
print("[FASE 6a] MATRIZ DE ANÁLISE DESCRITIVA")
print("="*80 + "\n")

# Criar matriz de análise por fácia
print("ANÁLISE COMPARATIVA ENTRE FÁCIAS:\n")

comparison_metrics = pd.DataFrame({
    'Métrica': ['Qtd. Artigos', 'Score Médio', 'Citações Médias', 
                'Após 2020 (%)', 'Com Abstract (%)', 'Com DOI (%)'],
    'Núcleo A': [
        len(df_analysis[df_analysis['fascia'] == 'Núcleo A']),
        df_analysis[df_analysis['fascia'] == 'Núcleo A']['score_final_100'].mean(),
        df_analysis[df_analysis['fascia'] == 'Núcleo A']['citations'].mean(),
        (df_analysis[df_analysis['fascia'] == 'Núcleo A']['year'] >= 2020).sum() / 
            len(df_analysis[df_analysis['fascia'] == 'Núcleo A']) * 100,
        (df_analysis[df_analysis['fascia'] == 'Núcleo A']['abstract'] != '').sum() / 
            len(df_analysis[df_analysis['fascia'] == 'Núcleo A']) * 100,
        (df_analysis[df_analysis['fascia'] == 'Núcleo A']['doi'] != '').sum() / 
            len(df_analysis[df_analysis['fascia'] == 'Núcleo A']) * 100,
    ],
    'Núcleo B': [
        len(df_analysis[df_analysis['fascia'] == 'Núcleo B']),
        df_analysis[df_analysis['fascia'] == 'Núcleo B']['score_final_100'].mean(),
        df_analysis[df_analysis['fascia'] == 'Núcleo B']['citations'].mean(),
        (df_analysis[df_analysis['fascia'] == 'Núcleo B']['year'] >= 2020).sum() / 
            len(df_analysis[df_analysis['fascia'] == 'Núcleo B']) * 100,
        (df_analysis[df_analysis['fascia'] == 'Núcleo B']['abstract'] != '').sum() / 
            len(df_analysis[df_analysis['fascia'] == 'Núcleo B']) * 100,
        (df_analysis[df_analysis['fascia'] == 'Núcleo B']['doi'] != '').sum() / 
            len(df_analysis[df_analysis['fascia'] == 'Núcleo B']) * 100,
    ],
    'Periférico': [
        len(df_analysis[df_analysis['fascia'] == 'Periférico']),
        df_analysis[df_analysis['fascia'] == 'Periférico']['score_final_100'].mean(),
        df_analysis[df_analysis['fascia'] == 'Periférico']['citations'].mean(),
        (df_analysis[df_analysis['fascia'] == 'Periférico']['year'] >= 2020).sum() / 
            len(df_analysis[df_analysis['fascia'] == 'Periférico']) * 100,
        (df_analysis[df_analysis['fascia'] == 'Periférico']['abstract'] != '').sum() / 
            len(df_analysis[df_analysis['fascia'] == 'Periférico']) * 100,
        (df_analysis[df_analysis['fascia'] == 'Periférico']['doi'] != '').sum() / 
            len(df_analysis[df_analysis['fascia'] == 'Periférico']) * 100,
    ]
})

print(comparison_metrics.to_string(index=False))

# Salvar como CSV também
comparison_metrics.to_csv('cientometria_comparativa_fascias.csv', index=False, encoding='utf-8')
print("\n✓ Matriz exportada: cientometria_comparativa_fascias.csv")

print("\n" + "="*80)
print("[FASE 6b] DISTRIBUIÇÃO TEMPORAL")
print("="*80 + "\n")

# Artigos por década
df_analysis['decade'] = (df_analysis['year'] // 10 * 10).astype(str) + 's'
decade_dist = df_analysis[df_analysis['decade'] != 'nans'].groupby('decade').agg({
    'title': 'count',
    'citations': 'mean',
    'score_final_100': 'mean'
}).rename(columns={'title': 'Quantidade'}).round(2)

print("Distribuição por período:\n")
print(decade_dist.to_string())

decade_dist.to_csv('cientometria_distribuicao_temporal.csv', encoding='utf-8')
print("\n✓ Distribuição temporal exportada: cientometria_distribuicao_temporal.csv")

print("\n" + "="*80)
print("[FASE 6c] ANÁLISE POR FONTE DE DADOS")
print("="*80 + "\n")

source_dist = df_analysis.groupby('source_db').agg({
    'title': 'count',
    'score_final_100': 'mean',
    'citations': 'mean',
}).rename(columns={'title': 'Quantidade'}).round(2).sort_values('Quantidade', ascending=False)

print("Contribuição por base de dados:\n")
print(source_dist.to_string())

source_dist.to_csv('cientometria_distribuicao_fontes.csv', encoding='utf-8')
print("\n✓ Distribuição por fonte exportada: cientometria_distribuicao_fontes.csv")

In [ ]:
# Corrigir a análise de fontes e criar RELATÓRIO FINAL CONSOLIDADO

print("\n" + "="*80)
print("[FASE 6c REVISADA] ANÁLISE POR FONTE DE DADOS")
print("="*80 + "\n")

# Verificar se há dados em source_db
print("Verificando coluna source_db:")
print(df_analysis['source_db'].value_counts())

# Se vazio, usar informações de db_type
if df_analysis['source_db'].isna().all():
    source_dist = df_analysis.groupby('db_type').agg({
        'title': 'count',
        'score_final_100': 'mean',
        'citations': 'mean',
    }).rename(columns={'title': 'Quantidade'}).round(2).sort_values('Quantidade', ascending=False)
else:
    source_dist = df_analysis.groupby('source_db').agg({
        'title': 'count',
        'score_final_100': 'mean',
        'citations': 'mean',
    }).rename(columns={'title': 'Quantidade'}).round(2).sort_values('Quantidade', ascending=False)

print("\nContribuição por base de dados:\n")
print(source_dist.to_string())

source_dist.to_csv('cientometria_distribuicao_fontes.csv', encoding='utf-8')
print("\n✓ Distribuição por fonte exportada: cientometria_distribuicao_fontes.csv")

# ============================================================================
# RELATÓRIO FINAL CONSOLIDADO
# ============================================================================

print("\n\n" + "="*80)
print("╔" + "="*78 + "╗")
print("║" + " "*78 + "║")
print("║" + "RELATÓRIO FINAL DE CIENTOMETRIA - ANÁLISE BIBLIOGRÁFICA".center(78) + "║")
print("║" + "Previsão de Enchentes com Inteligência Artificial e Machine Learning".center(78) + "║")
print("║" + " "*78 + "║")
print("╚" + "="*78 + "╝")

print("\n" + "="*80)
print("1. RESUMO EXECUTIVO")
print("="*80 + "\n")

print("""
A análise cientométrica foi realizada sobre 11.547 artigos únicos, consolidados
de 8 bases de dados (Lens, Scopus, IEEE, SciELO), resultando em:

  • 2.068 artigos relevantes (17.9% da base) com foco em previsão de inundações e IA/ML
  • 507 artigos recomendados para leitura (Núcleos A + B = 24.5%)
  • 13 artigos de máxima relevância (Núcleo A = 0.6%)
  • Redução de 90.3% do universo inicial para leitura viável
""")

print("="*80)
print("2. METODOLOGIA: SCORE DE RELEVÂNCIA")
print("="*80 + "\n")

print("""
EQUAÇÃO GERAL:
    Score_i = 0.35·C*_i + 0.20·R*_i + 0.30·T*_i + 0.15·A*_i

COMPONENTES:
    
    (1) C*_i - CITAÇÕES NORMALIZADAS (peso = 0.35)
        C*_i = (C_i - C_min) / (C_max - C_min)
        Normaliza citações para intervalo [0,1]
        Prioriza artigos com maior impacto científico

    (2) R*_i - RECÊNCIA NORMALIZADA (peso = 0.20)
        R*_i = (Y_i - Y_min) / (Y_max - Y_min)
        Normaliza ano de publicação para intervalo [0,1]
        Favorece pesquisa moderna (2020+)

    (3) T*_i - ADERÊNCIA TEMÁTICA (peso = 0.30)
        T*_i ∈ {0.4, 0.7, 1.0}
        Verifica presença de "flood prediction" E "AI/ML" em:
        - 1.0: Título + Abstract + Keywords (máxima aderência)
        - 0.7: Abstract/Keywords (moderada)
        - 0.4: Apenas um dos tópicos (baixa)

    (4) A*_i - ADERÊNCIA DADOS/MÉTODO/RESULTADO (peso = 0.15)
        A*_i = (I_i + M_i + O_i + C_i) / 4
        Onde:
          I_i = presença de dados hidrológicos/geoespaciais
          M_i = ML/DL explícito (LSTM, RNN, Random Forest, etc.)
          O_i = output de previsão/forecasting/mapping
          C_i = contexto urbano/costeiro/bacia
        Cada termo ∈ {0.0, 0.5, 1.0}
        Intervalo final: [0.5, 1.0]

ESCALA FINAL: Score normalizado para 0-100
""")

print("="*80)
print("3. CRITÉRIOS DE FILTRO TEMÁTICO")
print("="*80 + "\n")

print("""
FILTRO APLICADO:
    Todos os artigos analisados atendem aos critérios:
    
    ✓ Menção explícita de "inundação/cheia/flood" E "previsão/forecasting"
    ✓ Presença de "machine learning" OU "deep learning" OU "AI"
    ✓ Sem termos de exclusão (flood insurance, mythology, buffer overflow, etc.)
    ✓ Dados hidrológicos (chuva, vazão, nível) E/OU geoespaciais (satélite, DEM)
""")

print("="*80)
print("4. DEFINIÇÃO DE FÁCIAS")
print("="*80 + "\n")

print("""
┌─ NÚCLEO A (Score ≥ 70) ────────────────────────────────────────────────┐
│ Artigos: 13 (0.6% da base)                                              │
│ Perfil:  Altíssimo impacto (avg 283 citações), muito recentes           │
│ Leitura: PRIORITÁRIA - Devem ser lidos integralmente                    │
│ Tempo:   ~40-50 horas (3-4 horas/artigo)                                │
│                                                                           │
│ Exemplos:                                                                │
│ - A Rainfall-Runoff Model With LSTM (555 citações, 2020)               │
│ - Deep Learning with LSTM for Rainfall-Runoff (475 cit., 2018)        │
│ - Flash Flood Susceptibility w/ ML (239 cit., 2019)                    │
└──────────────────────────────────────────────────────────────────────────┘

┌─ NÚCLEO B (50 ≤ Score < 70) ──────────────────────────────────────────┐
│ Artigos: 494 (23.9% da base)                                            │
│ Perfil:  Bom impacto (avg 20 citações), muito recentes (92% pós-2020)  │
│ Leitura: SELETIVA - Ler títulos/abstracts, selecionar 50-100           │
│ Tempo:   ~50-100 horas (30min-1h por artigo selecionado)               │
│                                                                           │
│ Uso:     Fundamentar métodos específicos, comparar abordagens            │
└──────────────────────────────────────────────────────────────────────────┘

┌─ PERIFÉRICO (30 ≤ Score < 50) ────────────────────────────────────────┐
│ Artigos: 1.508 (72.9% da base)                                          │
│ Perfil:  Baixo impacto (avg 10 cit.), recentes (92% pós-2020)          │
│ Leitura: OPCIONAL - Usar apenas para preenchimento/lacunas             │
│ Tempo:   Máximo 20 horas (consultivo, não leitura integral)            │
│                                                                           │
│ Nota:    Muitos artigos legítimos mas contextos diferentes              │
└──────────────────────────────────────────────────────────────────────────┘

┌─ DESCARTE (Score < 30) ───────────────────────────────────────────────┐
│ Artigos: 53 (2.6% da base)                                              │
│ Perfil:  Marginal à sua pesquisa                                        │
│ Leitura: NÃO RECOMENDADO                                                │
└──────────────────────────────────────────────────────────────────────────┘
""")

print("\n" + "="*80)
print("5. RECOMENDAÇÃO DE TEMPO DE LEITURA")
print("="*80 + "\n")

print("""
CENÁRIO 1 - RIGOROSO (apenas Núcleo A)
    Artigos:  13
    Tempo:    40-50 horas (~4 semanas @ 10h/sem)
    Cobertura: 0.6% da base
    Resultado: Fundamentação muito sólida

CENÁRIO 2 - EQUILIBRADO (Núcleos A + 50 do B)
    Artigos:  63
    Tempo:    70-100 horas (~8 semanas @ 10h/sem)
    Cobertura: 3% da base
    Resultado: Fundamentação sólida + métodos variados

CENÁRIO 3 - AMPLO (Núcleos A + B inteiro)
    Artigos:  507
    Tempo:    200-300 horas (~6 meses @ 10h/sem)
    Cobertura: 24.5% da base
    Resultado: Visão abrangente + tendências atuais

RECOMENDAÇÃO PARA VOCÊ: Cenário 2 ou 3
    • Seu mestrado requer fundamentação sólida
    • Dois anos disponíveis
    • Leitura estimada: 100-150 artigos é viável
    • Comece com Núcleo A (garantido), expanda conforme necessário
""")

print("\n" + "="*80)
print("6. PRÓXIMOS PASSOS PRÁTICOS")
print("="*80 + "\n")

print("""
ARQUIVOS GERADOS PARA VOCÊ:

1. cientometria_nucleoa_recomendado.csv
   → 13 artigos do Núcleo A (LEIA ESTES!)
   → Ordernados por Score
   → Inclui DOI para acesso fácil

2. cientometria_nucleos_ab.csv
   → 507 artigos de Núcleos A + B
   → Para seleção detalhada
   → Colunas: Título, Ano, Citações, Score, DOI

3. cientometria_analise_completa.csv
   → 2.068 artigos da análise
   → Para consultas específicas
   → Todas as componentes do score

AÇÕES RECOMENDADAS:

□ Passo 1: Abrir cientometria_nucleoa_recomendado.csv
          Ler todos os 13 artigos (40-50h)

□ Passo 2: Após Núcleo A, abrir cientometria_nucleos_ab.csv
          Filtrar por suas palavras-chave específicas
          Selecionar 50-100 artigos do Núcleo B
          
□ Passo 3: Usar esses dados para:
          - Estado da arte
          - Metodologia (qual algoritmo usar)
          - Dados de entrada (qual sensor/fonte)
          - Métricas de avaliação (qual loss/accuracy medir)

□ Passo 4: Documentar suas escolhas
          "Selecionei 13 artigos do Núcleo A por (...)
           Selecionei 75 do Núcleo B porque (...)"
          Isso demonstra rigor científico

□ Passo 5: (Opcional) Usar periférico para preencher lacunas
          "Encontrei 8 artigos adicionais sobre [tópico específico]"
""")

print("\n" + "="*80)
print("7. ESTATÍSTICAS FINAIS DA ANÁLISE")
print("="*80 + "\n")

stats_text = f"""
BASE ORIGINAL:
    Lens (EN)....................... 10.840 artigos
    Scopus (todos).................. 5.120 artigos
    IEEE Xplore.................... 612 artigos
    SciELO (todos).................. 5 artigos
    ─────────────────────────────────
    TOTAL INICIAL: 16.845 artigos

APÓS DEDUPLICAÇÃO:
    Base consolidada................ 11.547 artigos únicos
    Redução...................... 31.5% (duplicatas removidas)

APÓS FILTRO TEMÁTICO:
    Núcleo Expandido (mod. rigoroso). 2.068 artigos (17.9%)
    Removidos por exclusão.......... 1.676 artigos
    Fora do escopo.................. 7.803 artigos

DISTRIBUIÇÃO FINAL:
    Núcleo A (máxima relevância).... 13 artigos (0.6%)
    Núcleo B (boa relevância)....... 494 artigos (23.9%)
    Periférico (relevância baixa)... 1.508 artigos (72.9%)
    Descarte....................... 53 artigos (2.6%)

IMPACTO:
    Redução total: 11.547 → 507 artigos (recomendados)
    Percentual reduzido: 95.6% filtrado
    Precisão esperada: Muito alta (foco exato em sua pesquisa)

PERÍODO COBERTO:
    1992 - 2026 (34 anos de pesquisa)
    Concentração: 2020-2026 (1.856 artigos = 89.8% do Núcleo Expandido)
    Tendência: Crescimento exponencial em ML para flood desde 2015

CITAÇÕES (NÚCLEO A):
    Mínimo: 143 citações
    Máximo: 555 citações
    Média: 283 citações
    → Estes são trabalhos ALTAMENTE CITADOS e confiáveis
"""

print(stats_text)

# Salvar relatório como arquivo de texto
with open('cientometria_relatorio_final.txt', 'w', encoding='utf-8') as f:
    f.write("="*80 + "\n")
    f.write("RELATÓRIO FINAL DE CIENTOMETRIA\n")
    f.write("Previsão de Enchentes com IA/ML - Dissertação de Mestrado\n")
    f.write("="*80 + "\n\n")
    f.write(stats_text)

print("\n✓ Relatório salvo: cientometria_relatorio_final.txt")

In [ ]:
# Criar uma tabela-resumo final para visualização rápida

print("\n" + "╔" + "="*78 + "╗")
print("║" + "RESUMO FINAL - ANÁLISE CIENTOMÉTRICA COMPLETA".center(78) + "║")
print("╚" + "="*78 + "╝\n")

resumo_data = {
    'MÉTRICA': [
        'Base Original (consolidada)',
        'Duplicatas Removidas',
        '→ Artigos Únicos',
        '',
        'Filtro Temático (FP+AI+NOT_excl+Data)',
        '→ Núcleo Expandido',
        '',
        'Scoring & Fácias',
        '  • Núcleo A (Score ≥70)',
        '  • Núcleo B (50≤Score<70)',
        '  • Periférico (30≤Score<50)',
        '  • Descarte (Score<30)',
        '  → TOTAL ANALISADO',
        '',
        'Recomendação de Leitura',
        '  • Prioritária (Núcleo A)',
        '  • Seletiva (Núcleo A+50B)',
        '  • Ampla (Núcleos A+B)',
        '',
        'Tempo Estimado de Leitura',
        '  • Núcleo A',
        '  • Núcleo A + 50 do B',
        '  • Núcleos A + B completo',
    ],
    'VALOR': [
        '16.845',
        '5.298 (31.5%)',
        '11.547',
        '',
        '(FP+AI/ML + Hydro/Geo + NOT_excl)',
        '2.068 (17.9% da base)',
        '',
        '',
        '13 (0.6%)',
        '494 (23.9%)',
        '1.508 (72.9%)',
        '53 (2.6%)',
        '2.068',
        '',
        '',
        '13 artigos',
        '63 artigos',
        '507 artigos',
        '',
        '',
        '40-50 horas',
        '70-100 horas',
        '200-300 horas',
    ]
}

import pandas as pd
df_resumo = pd.DataFrame(resumo_data)
print(df_resumo.to_string(index=False))

print("\n" + "="*80)
print("EQUAÇÃO DO SCORE - RESUMO VISUAL")
print("="*80 + "\n")

eq_visual = """
╔════════════════════════════════════════════════════════════════════════════╗
║                        Score_i = 0.35·C*_i + 0.20·R*_i + 0.30·T*_i + 0.15·A*_i  ║
╚════════════════════════════════════════════════════════════════════════════╝

    35%                    20%                   30%                 15%
    │                      │                     │                    │
    ├─ Citações            ├─ Recência           ├─ Temática          ├─ Dados/Método
    │  (0 até 555)         │  (1992 até 2026)    │  (0.4, 0.7, 1.0)   │  (0.5 até 1.0)
    │                      │                     │                    │
    │  Impacto             │  Modernidade        │  Aderência FP+AI   │  Input+Method
    │  Científico          │  da Pesquisa        │  às keywords       │  +Output+Context
    │                      │                     │                    │
    └─────────┬────────────┴─────────┬───────────┴─────────┬──────────┴──────────┘
              │                      │                     │
              └──────────────────────┴─────────────────────┴─ Normaliza para [0-100]

INTERPRETAÇÃO:
• Um artigo é relevant se tem ALTO SCORE em TODOS os 4 componentes
• Um artigo mediocre aparece no Periférico (score 30-50)
• Um artigo excepcional sobe para Núcleo A (score ≥70)
"""

print(eq_visual)

print("\n" + "="*80)
print("CHECKLIST FINAL PARA VOCÊ")
print("="*80 + "\n")

checklist = """
✓ FASE 1: COLETA
  ✓ Bases carregadas: Lens (EN+PT), Scopus (3x), IEEE, SciELO
  ✓ Total consolidado: 11.547 artigos únicos
  ✓ Duplicatas removidas: 5.298

✓ FASE 2: NORMALIZAÇÃO
  ✓ Campos harmonizados: título, ano, abstract, keywords, citações, DOI
  ✓ Estrutura uniforme em todas as bases
  ✓ Pronto para análise quantitativa

✓ FASE 3: FILTRO TEMÁTICO
  ✓ Critério: Flood Prediction AND AI/ML AND (Hydro OR Geo) AND NOT_excluded
  ✓ Base reduzida a 2.068 artigos relevantes (17.9%)
  ✓ Qualidade: Todos mencionam explicitamente sua área

✓ FASE 4: SCORE
  ✓ Equação com 4 componentes + 4 pesos bem definidos
  ✓ Todos calculados e normalizados [0, 100]
  ✓ Interpretável: sabe por que um artigo tem score X

✓ FASE 5: FÁCIAS
  ✓ 13 artigos no Núcleo A (máxima prioridade)
  ✓ 494 artigos no Núcleo B (boa qualidade)
  ✓ 1.508 no Periférico (complementar)
  ✓ 53 no Descarte (não relevante)

✓ FASE 6: EXPORTAÇÃO
  ✓ cientometria_nucleoa_recomendado.csv (13 artigos - COMECE AQUI!)
  ✓ cientometria_nucleos_ab.csv (507 artigos - seleção ampla)
  ✓ cientometria_analise_completa.csv (2.068 artigos - referência)
  ✓ cientometria_relatorio_final.txt (este relatório em texto)

✓ PRÓXIMO PASSO
  ✓ Baixe o arquivo "cientometria_nucleoa_recomendado.csv"
  ✓ Abra cada um dos 13 artigos (procure por DOI ou título no Google Scholar)
  ✓ Leia com atenção e resuma em uma planilha sua própria
  ✓ Identifique: métodos, dados, resultados
  ✓ Depois selecione 50-100 do Núcleo B para aprofundar

IMPORTANTE:
→ Você tem 13 artigos GARANTIDOS para ler
→ Cada um vale ~40-50 horas de trabalho
→ Ao final de todos, você terá fundação sólida para sua dissertação
→ Use os do Núcleo B para preencher gaps específicos
→ Use Periférico só se tiver tempo ou para contexto muito específico
"""

print(checklist)

print("\n" + "="*80)
print("DÚVIDAS FREQUENTES")
print("="*80 + "\n")

faq = """
P: Por que só 13 artigos no Núcleo A?
R: Porque combinamos RIGOR (FP + AI/ML + dados específicos) com IMPACTO 
   (citações altas = confiáveis). Esses 13 são a "elite" do tópico.

P: Posso confiar na equação do score?
R: Sim. É uma combinação linear validada de métricas estabelecidas 
   (citações, recência, temática). Mas é preferência minha - você pode ajustar
   os pesos se quiser dar mais/menos peso a citações ou recência.

P: O que fazer com os 1.500+ artigos periféricos?
R: Guarde para consulta. Se descobrir uma lacuna específica (ex: "só encontrei
   3 artigos sobre flood em estuários"), use o periférico como repositório.

P: Preciso ler todos os 507 (A+B)?
R: Não! Leia: Núcleo A (13) inteiro + seleção de 50-100 do B baseado em 
   seus tópicos específicos (verifique título/abstract antes).

P: Como "selecionei" os 507 do Núcleo A+B?
R: Score ≥ 50. Significa que em TODOS os critérios (temática, dados, método,
   recência) teve pelo menos nota "boa". Não é aleatório.

P: As equações estão certas?
R: Sim. Cada uma é padrão em cientometria (normalização min-max, escala 0-1,
   ponderada). Mas sinta-se à vontade para ajustar pesos se tiver preferência.
"""

print(faq)

print("\n" + "="*80)
print("FIM DA ANÁLISE CIENTOMÉTRICA")
print("="*80)
print("\nArquivos prontos para download. Boa leitura! 📚\n")

In [ ]:
# Criar um sumário dos TOP 50 artigos para referência rápida

print("\n" + "="*80)
print("GERANDO SUMÁRIO DOS TOP 50 ARTIGOS RECOMENDADOS")
print("="*80 + "\n")

top50 = df_export.head(50).copy()
top50['rank'] = range(1, len(top50) + 1)

# Reordenar colunas para melhor visualização
top50_display = top50[['rank', 'fascia', 'title', 'year', 'citations', 'score_final_100', 'doi']].copy()
top50_display.columns = ['Rank', 'Fácia', 'Título', 'Ano', 'Citações', 'Score', 'DOI']

top50_display.to_csv('cientometria_top50_recomendados.csv', index=False, encoding='utf-8')

print("TOP 50 ARTIGOS RECOMENDADOS\n")
print(top50_display.to_string(index=False))

print("\n✓ Arquivo exportado: cientometria_top50_recomendados.csv")

# Criar um sumário em formato tabular simples para fácil leitura
print("\n" + "="*80)
print("SUMÁRIO EXECUTIVO - RESULTADO FINAL")
print("="*80 + "\n")

summary_final = f"""
┌──────────────────────────────────────────────────────────────────────────────┐
│                     ANÁLISE CIENTOMÉTRICA - RESULTADO FINAL                  │
└──────────────────────────────────────────────────────────────────────────────┘

1. BASES DE DADOS CONSULTADAS:
   • Lens (Inglês + Português).......... 11.108 artigos
   • Scopus (3 variações)............... 5.120 artigos
   • IEEE Xplore........................ 612 artigos
   • SciELO (Inglês + Português)........ 5 artigos
   ─────────────────────────────────────────────
   TOTAL BRUTO: 16.845 artigos

2. PROCESSAMENTO:
   • Deduplicação...................... 5.298 removidos (31,5%)
   • Base consolidada.................. 11.547 únicos
   • Filtro temático aplicado.......... 2.068 relevantes (17,9%)

3. ANÁLISE DE RELEVÂNCIA:
   Equação: Score = 0.35·C* + 0.20·R* + 0.30·T* + 0.15·A*
   
   Componentes:
   • C* (Citações, 0-555).............. Peso: 35%
   • R* (Recência, 1992-2026).......... Peso: 20%
   • T* (Temática, 0.4-1.0)............ Peso: 30%
   • A* (Dados/Método, 0.5-1.0)........ Peso: 15%

4. FÁCIAS RESULTANTES:
   ┌─────────────────┬──────────┬─────────┬──────────┬──────────────────┐
   │     FÁCIA       │ ARTIGOS  │    %    │  SCORE   │      PERFIL      │
   ├─────────────────┼──────────┼─────────┼──────────┼──────────────────┤
   │ Núcleo A        │   13     │  0,6%   │  ≥70     │ Máxima relevância│
   │ Núcleo B        │  494     │ 23,9%   │ 50-70    │ Boa relevância   │
   │ Periférico      │ 1.508    │ 72,9%   │ 30-50    │ Baixo impacto    │
   │ Descarte        │   53     │  2,6%   │ <30      │ Marginal         │
   ├─────────────────┼──────────┼─────────┼──────────┼──────────────────┤
   │ TOTAL ANALISADO │ 2.068    │ 100%    │          │                  │
   └─────────────────┴──────────┴─────────┴──────────┴──────────────────┘

5. RECOMENDAÇÃO DE LEITURA:
   
   Opção 1 (Rigorosa):
   • 13 artigos (Núcleo A)
   • 40-50 horas
   • Cobertura: 0,6%
   
   Opção 2 (Equilibrada) ⭐ RECOMENDADA
   • 63 artigos (13 A + 50 B)
   • 70-100 horas
   • Cobertura: 3,0%
   
   Opção 3 (Ampla):
   • 507 artigos (Núcleos A+B)
   • 200-300 horas
   • Cobertura: 24,5%

6. PRÓXIMOS PASSOS:
   □ Download: cientometria_nucleoa_recomendado.csv (13 artigos)
   □ Download: cientometria_nucleos_ab.csv (507 artigos)
   □ Download: relatorio_cientometria.md (metodologia completa)
   □ Leitura: Núcleo A primeiro (garantido)
   □ Seleção: 50-100 do Núcleo B conforme tópicos específicos
   □ Consulta: Periférico apenas para lacunas muito específicas

7. ARQUIVOS GERADOS:
   ✓ cientometria_nucleoa_recomendado.csv ........... 13 artigos (PRIORITÁRIO)
   ✓ cientometria_nucleos_ab.csv ................... 507 artigos (RECOMENDADO)
   ✓ cientometria_top50_recomendados.csv ........... 50 top ranked
   ✓ cientometria_analise_completa.csv ............ 2.068 artigos
   ✓ relatorio_cientometria.md .................... Metodologia completa
   ✓ cientometria_comparativa_fascias.csv ........ Matriz comparativa
   ✓ cientometria_distribuicao_temporal.csv ...... Tendências 1990-2026
   ✓ cientometria_relatorio_final.txt ............ Sumário em texto

╔──────────────────────────────────────────────────────────────────────────────╗
║                                                                              ║
║ RESULTADO: Redução de 95,6% do universo de pesquisa mantendo máxima         ║
║ relevância temática e qualidade científica.                                 ║
║                                                                              ║
║ Você pode começar a ler com CONFIANÇA: os 13 artigos do Núcleo A           ║
║ representam o topo da pesquisa atual em previsão de enchentes com IA/ML.   ║
║                                                                              ║
╚──────────────────────────────────────────────────────────────────────────────╝

Data da análise: Janeiro 2026
Analista: Sistema de Cientometria Automática
Instituição: UNIVALI - Programa de Pós-Graduação em Computação Aplicada
"""

print(summary_final)

# Salvar sumário final
with open('COMECE_AQUI.txt', 'w', encoding='utf-8') as f:
    f.write(summary_final)

print("\n✓ Sumário executivo salvo: COMECE_AQUI.txt")

print("\n" + "="*80)
print("ANÁLISE CIENTÍFOMÉTRICA CONCLUÍDA COM SUCESSO!")
print("="*80)
print(f"""
Total de arquivos gerados: 8
Tempo de processamento: ~2-3 minutos
Base de dados consolidada: 11.547 artigos
Base de análise final: 2.068 artigos
Recomendações de leitura: 507 artigos
Prioridade máxima: 13 artigos

Próxima ação: Abra o arquivo 'COMECE_AQUI.txt' para instruções
""")

## 5. Conferência: o resultado reproduz o relatório original?

In [ ]:
cons_pre = pd.concat(all_dfs, ignore_index=True)
sem_doi = int((cons_pre['doi'] == '').sum())
apos_doi = len(cons_pre) - sem_doi - int(cons_pre[cons_pre['doi'] != ''].duplicated(subset=['doi']).sum())
dup_doi = len(cons_pre) - sem_doi - apos_doi
obtido = {
  'registros nas exportações': len(cons_pre),
  'removidos por DOI (declarado)': dup_doi + sem_doi,
  'removidos por título (declarado)': apos_doi - len(df_consolidated),
  'removidos no total (declarado)': len(cons_pre) - len(df_consolidated),
  'únicos após deduplicação': len(df_consolidated),
  'corpus (núcleo expandido)': len(df_expanded),
  **{k: int((df_analysis['fascia']==k).sum()) for k in ['Núcleo A','Núcleo B','Periférico','Descarte']},
}
print(f"{'item':45s}{'declarado':>10s}{'obtido':>10s}")
for k,v in DECLARADO.items():
    ok = '✓' if obtido[k]==v else '✗'
    print(f"{k:45s}{v:>10d}{obtido[k]:>10d}  {ok}")
print()
print(f"Detalhe da remoção por DOI: {dup_doi} duplicatas reais + {sem_doi} registros SEM DOI descartados = {dup_doi+sem_doi}")
print("Obs.: a divisão 4.233/1.065 do relatório não corresponde à execução; o total (5.298) sim.")

In [ ]:
nucleoA = df_analysis[df_analysis.fascia=='Núcleo A'].sort_values('score_final_100', ascending=False)
nucleoA[['doi','year','citations','score_final_100']].assign(titulo=nucleoA['title'].str[:80])

## 6. Diagnóstico das falhas do procedimento original

In [ ]:
import re
print("== (a) Colunas mapeadas que NÃO existem no arquivo (caem em 'Unknown'/vazio) ==")
for nome, df, chave in [('Lens (EN)',lens_en,'lens'),('Scopus (sem filtro)',scopus_all,'scopus'),
                        ('IEEE',ieee,'ieee'),('SciELO (EN)',scielo_en,'scielo')]:
    ausentes = [v for k,v in field_mapping[chave].items() if v and v not in df.columns and k!='field']
    print(f"  {nome:22s} {len(df):>6} registros | ausentes: {ausentes}")

print("\n== (b) Contribuição efetiva de cada base ao corpus consolidado ==")
print("  source_db preenchido:", int(df_consolidated['source_db'].notna().sum()), "de", len(df_consolidated), "(coluna vazia)")
print("  títulos ausentes (NaN/'Unknown') antes da dedup (IEEE sem coluna Title):", int((cons_pre['title'].isna()|(cons_pre['title']=='Unknown')).sum()))
print("  títulos ausentes após a dedup:", int((df_consolidated['title'].isna()|(df_consolidated['title']=='Unknown')).sum()))
print("  db_type no corpus final:", df_analysis['db_type'].value_counts().to_dict())

print("\n== (c) Exclusão por trecho de texto ('dam' casa com 'damage') ==")
cand = df_consolidated[(df_consolidated.flood_pred_score==1)&(df_consolidated.ai_ml_score==1)&
                       ((df_consolidated.hydro_score==1)|(df_consolidated.geospatial_score==1))]
w = re.compile(r'\bdams?\b|\bdikes?\b|\blevees?\b', re.I)
def so_substring(r):
    t = (str(r['title'])+' '+str(r['abstract'])).lower()
    outros = [k for k in keywords_exclude if k not in ('dam','dike','levee') and k in t]
    return (not outros) and any(k in t for k in ('dam','dike','levee')) and not w.search(t)
n_sub = int(cand[cand['is_excluded']].apply(so_substring, axis=1).sum())
print(f"  candidatos que passariam sem a exclusão: {len(cand)}")
print(f"  excluídos: {int(cand['is_excluded'].sum())}  | excluídos SÓ por 'dam/dike/levee' dentro de outra palavra: {n_sub} ({n_sub/len(cand):.1%})")
mos = df_consolidated[df_consolidated['doi'].str.lower().str.contains('w10111536', na=False)]
print("  Mosavi et al. 2018 (10.3390/w10111536) na base consolidada:", len(mos), "| excluído:", bool(mos['is_excluded'].iloc[0]) if len(mos) else None)
if len(mos):
    t = (str(mos.iloc[0]['title'])+' '+str(mos.iloc[0]['abstract'])).lower()
    print("    termos de exclusão presentes:", [k for k in keywords_exclude if k in t])

## 7. Teste de robustez (não é ajuste do método)
**7.1 Variante com a exclusão corrigida.** Muda *apenas* a exclusão `dam/dike/levee` (de trecho de texto para palavra inteira); todo o resto é idêntico. O objetivo não é "chegar" a um conjunto, e sim medir quanto o resultado depende dessa decisão e do corte fixo de 70.

In [ ]:
orig13 = set(df_analysis[df_analysis.fascia=='Núcleo A']['doi'].str.lower())
EX2 = [k for k in keywords_exclude if k not in ('dam','dike','levee')]
def excl2(t):
    t = str(t).lower(); return any(k in t for k in EX2) or bool(w.search(t))
dv = df_consolidated.copy()
dv['is_excluded2'] = (dv['title'].astype(str)+' '+dv['abstract'].astype(str)).map(excl2)
b = dv[(dv.flood_pred_score==1)&(dv.ai_ml_score==1)&((dv.hydro_score==1)|(dv.geospatial_score==1))&(~dv.is_excluded2)].copy()
mx2, mn2 = b['citations'].max(), b['citations'].min()
b['C_norm'] = ((b['citations']-mn2)/(mx2-mn2)).fillna(0)
z0, z1 = b['year'][b['year'].notna()].min(), b['year'].max()
b['R_norm'] = b['year'].apply(lambda y: (y-z0)/(z1-z0) if pd.notna(y) and y>=z0 else 0)
b['T_score'] = b.apply(classify_thematic_adherence, axis=1)
_has = check_keyword_presence
b['A_score'] = (b.apply(lambda r: 1.0 if _has(str(r['abstract'])+' '+str(r['keywords']), input_keywords) else 0.0, axis=1)
              + b.apply(lambda r: 1.0 if _has(str(r['abstract'])+' '+str(r['title']), ml_specific) else 0.5, axis=1)
              + b.apply(lambda r: 1.0 if _has(str(r['title'])+' '+str(r['abstract']), output_keywords) else 0.5, axis=1)
              + b.apply(lambda r: 1.0 if _has(str(r['abstract'])+' '+str(r['keywords']), context_keywords) else 0.5, axis=1)) / 4
b['sf'] = w1*b.C_norm + w2*b.R_norm + w3*b.T_score + w4*b.A_score
b['s100'] = (b.sf-b.sf.min())/(b.sf.max()-b.sf.min())*100
nA2 = b[b.s100>=70].sort_values('s100', ascending=False)
print(f"VARIANTE: corpus {len(b)} | Núcleo A {len(nA2)} | Núcleo B {int(((b.s100>=50)&(b.s100<70)).sum())} | máx. citações {mx2}")
print("Dos 13 originais, continuam no Núcleo A:", len(orig13 & set(nA2.doi.str.lower())))
ranks = sorted(int((b.s100 > b[b.doi.str.lower()==x].s100.iloc[0]).sum())+1 for x in orig13 if (b.doi.str.lower()==x).any())
print("Posição (ranking) dos 13 originais na variante:", ranks, f"de {len(b)}")
tab = []
for x in orig13:
    m = b[b.doi.str.lower()==x]
    o = df_analysis[df_analysis.doi.str.lower()==x]
    tab.append({'doi':x, 'escore_original':round(float(o.score_final_100.iloc[0]),1),
                'escore_variante':round(float(m.s100.iloc[0]),1) if len(m) else None,
                'posicao_variante':int((b.s100>m.s100.iloc[0]).sum())+1 if len(m) else None})
print("\nOs 13 originais na variante (escore original, escore e posição na variante):")
print(pd.DataFrame(tab).sort_values('posicao_variante').to_string(index=False))
print("\nTop 10 da variante:")
b.sort_values('s100', ascending=False).head(10)[['doi','year','citations','s100']].round(1)

**Leitura:** o corte absoluto de 70 é sensível à composição do corpus (um único artigo muito citado reescala todos os demais). A *posição* no ranking é bem mais estável que o *rótulo* "Núcleo A". Para a leitura prioritária, recomenda-se reportar os dois: o rótulo do procedimento original e a posição no ranking.

**7.2 Sensibilidade aos pesos** (10.000 sorteios com `seed` fixa; cada peso varia ±15%; também Dirichlet(5) e pesos iguais). Duas medidas: (i) **ranking** — quantos dos 13 originais continuam entre os 13 mais bem pontuados; (ii) **corte fixo** — quantos continuam com escore ≥ 70 após a renormalização min–max.

In [ ]:
rng = np.random.default_rng(42)
base_w = np.array([w1,w2,w3,w4]); X = df_analysis[['C_norm','R_norm','T_score','A_score']].values.astype(float)
pos_orig = set(np.where(df_analysis['fascia'].values=='Núcleo A')[0])
def medir(ww):
    s = X@ww; s100 = (s-s.min())/(s.max()-s.min())*100
    top13 = set(np.argsort(-s)[:13]); corte = set(np.where(s100>=70)[0])
    return len(top13&pos_orig), len(corte&pos_orig), len(corte)
def resumo(nome, gen, n=10000):
    r = np.array([medir(gen()) for _ in range(n)])
    print(f"{nome:16s} | ranking: média {r[:,0].mean():.1f}/13, mín {r[:,0].min()}, todos os 13 em {np.mean(r[:,0]==13):.1%}, ≥11 em {np.mean(r[:,0]>=11):.1%}"
          f" | corte 70: média {r[:,1].mean():.1f}/13, tamanho do Núcleo A {r[:,2].min()}–{r[:,2].max()}")
resumo('±15% uniforme', lambda: base_w*(1+rng.uniform(-.15,.15,4)))
resumo('Dirichlet(5)',  lambda: rng.dirichlet([5]*4))
print('pesos iguais     | (ranking, corte70, tamanho):', medir(np.array([.25]*4)))
print('um peso por vez ±15%:', [medir(np.where(np.arange(4)==i, base_w*(1+sg*.15), base_w)) for i in range(4) for sg in (-1,1)])

## 8. Validação externa de cobertura (22 referências internacionais de Dias et al., 2025)
Para cada referência: foi recuperada nas exportações? sobreviveu à deduplicação? passou no filtro temático? foi excluída por termo de exclusão? entrou no corpus? Combina por similaridade de título (≥ 0,90).

In [ ]:
try:
    from rapidfuzz import fuzz, process
except ImportError:
    import subprocess, sys; subprocess.run([sys.executable,'-m','pip','install','-q','rapidfuzz'])
    from rapidfuzz import fuzz, process
def nt(t): return re.sub(r"\s+"," ",re.sub(r"[^a-z0-9 ]"," ",str(t).lower())).strip()
REFS = {1:"A Brief review of flood forecasting techniques and their applications",
2:"A Critical Review of Emerging Technologies for Flash Flood Prediction Examining Artificial Intelligence Machine Learning Internet of Things Cloud Computing and Robotics Techniques",
3:"Flood Prediction Using Machine Learning Models Literature Review",
9:"Neurocomputing in Surface Water Hydrology and Hydraulics A Review of Two Decades Retrospective Current Status and Future Prospects",
10:"Exploring a Long Short-Term Memory based Encoder-Decoder framework for multi-step-ahead flood forecasting",
11:"Water Level Prediction Model Based on GRU and CNN",
12:"Designing Deep-Based Learning Flood Forecast Model With ConvLSTM Hybrid Algorithm",
13:"CRML A Convolution Regression Model With Machine Learning for Hydrology Forecasting",
14:"River Flooding Forecasting and Anomaly Detection Based on Deep Learning",
15:"FLOODWALL A Real-Time Flash Flood Monitoring and Forecasting System Using IoT",
16:"IoT-Enabled Flood Severity Prediction via Ensemble Machine Learning Models",
17:"Prediction of Flood Severity Level via Processing IoT Sensor Data Using a Data Science Approach",
18:"Digital Twin-assisted Fuzzy Logic-inspired Intelligent Approach for Flood Prediction",
19:"FFM Flood Forecasting Model Using Federated Learning",
20:"Sensor Networks Data Processing and Inference The Hydrology Challenge",
21:"An ensemble neural network model for real-time prediction of urban floods",
22:"Deep Learning Based Modal for Predicting Flooding Pattern and Climate Shifts",
28:"Recent Trends in the Quality Module of Sustainable Water Management Models A Systematic Review",
29:"Prediction of Flooding Due to Heavy Rainfall in India Using Machine Learning Algorithms Providing Advanced Warning",
30:"Prior Recognition of Flash Floods Concrete Optimal Neural Network Configuration Analysis for Multi-Resolution Sensing",
31:"Developments and Trends in Water Level Forecasting Using Machine Learning Models A Review",
32:"A review on applications of urban flood models in flood mitigation strategies"}
cp = cons_pre.copy(); cp['tn'] = cp['title'].map(nt)
cons_tn = set(df_consolidated['title'].map(nt)); cons_doi = set(df_consolidated['doi'].str.lower())
corpus_tn = set(df_analysis['title'].map(nt)); corpus_doi = set(df_analysis['doi'].str.lower())
cand_tn = set(cand['title'].map(nt))
linhas = []
for k, t in REFS.items():
    r = process.extractOne(nt(t), cp['tn'].tolist(), scorer=fuzz.ratio)
    if not r or r[1] < 90: linhas.append((k,'não recuperada','','','','')); continue
    row = cp.iloc[r[2]]; tn_ = row['tn']; doi_ = str(row['doi']).lower()
    sobreviveu = (tn_ in cons_tn) or (doi_ and doi_ in cons_doi)
    d_rows = df_consolidated[df_consolidated['title'].map(nt)==tn_]
    passou = bool(len(d_rows) and ((d_rows.flood_pred_score==1)&(d_rows.ai_ml_score==1)&((d_rows.hydro_score==1)|(d_rows.geospatial_score==1))).any())
    excl = bool(len(d_rows) and d_rows.is_excluded.any())
    inc = (tn_ in corpus_tn) or (doi_ and doi_ in corpus_doi)
    fx = df_analysis[df_analysis['title'].map(nt)==tn_]['fascia'].iloc[0] if inc and (df_analysis['title'].map(nt)==tn_).any() else ''
    linhas.append((k,'sim' if sobreviveu else 'perdida na dedup','passa' if passou else 'não passa','EXCLUÍDA' if excl else '', 'sim' if inc else 'não', fx))
res = pd.DataFrame(linhas, columns=['ref','recuperada/dedup','filtro temático','exclusão','no corpus','fácia'])
print(res.to_string(index=False))
print("\nrecuperadas:", int((res['recuperada/dedup']!='não recuperada').sum()), "de", len(REFS),
      "| no corpus:", int((res['no corpus']=='sim').sum()),
      "| barradas por termo de exclusão:", int((res['exclusão']=='EXCLUÍDA').sum()))

## 9. Síntese
- **Reprodutibilidade:** o procedimento original reproduz exatamente os números 16.845 → 11.547 → 2.068 → 13/494/1.508/53 (seção 5). A divisão 4.233/1.065 do relatório não corresponde à execução.
- **Falhas documentadas (seção 6):** colunas de IEEE/SciELO/Scopus não mapeadas; registros sem DOI descartados como se fossem duplicatas; proveniência (`source_db`) perdida; exclusão por trecho de texto remove artigos relevantes (ex.: Mosavi et al., 2018).
- **Robustez (seção 7):** os 13 do Núcleo A são estáveis a variações de pesos (7.2), mas o *rótulo* depende do corte fixo de 70 e da composição do corpus (7.1); a posição no ranking é mais estável.
- **Limite do método:** triagem por termos, sem validação humana. Os 13 artigos devem ser conferidos manualmente antes de serem tratados como leitura-base.